In [ ]:
from typing import Annotated, TypedDict

from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, BaseMessage

from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

from dotenv import load_dotenv

In [36]:
load_dotenv()

True

In [37]:
llm = ChatGroq(model = 'openai/gpt-oss-safeguard-20b')

In [38]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [39]:
def chat_node(state: ChatState):
    
    decision = interrupt({
        'type':'approval',
        'reason':'Model is about to answer a user question.',
        'question':state['messages'][-1].content,
        'instruction':'Approve this question? yes/no'
    })
    
    if decision['approved'] == 'no':
        return {'messages':[AIMessage(content = 'Not Approved.')]}
    
    else:
        response = llm.invoke(state['messages'])
        return {'messages':[response]}

In [40]:
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer = MemorySaver()

# Compile the app
app = builder.compile(checkpointer=checkpointer)

In [41]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

# STEP 1: user asks a question
initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

In [42]:
result

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='3b788c6f-c55c-4c5c-aa96-1fb59ae6eaba')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question? yes/no'}, id='4eb1b7365d7cb3bc1073d5266c2707c8')]}

In [43]:
message = result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'Explain gradient descent in very simple terms.',
 'instruction': 'Approve this question? yes/no'}

In [44]:
user_input = input(f"\nBackend message -{message} \n Approve this question? (y/n): ")

In [45]:
# Resume the graph
final_result = app.invoke(
    Command(resume = {'approved':user_input}),
    config = config,
)

In [46]:
print(final_result)

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='3b788c6f-c55c-4c5c-aa96-1fb59ae6eaba'), AIMessage(content="### Imagine you’re standing on a big, fuzzy hill\n\n* **Goal** – get to the lowest point (the valley).\n* **What you can see** – at your feet, you can feel whether the ground is going up or down (the *slope*), but you can’t see the whole hill.\n\n#### 1. Pick a starting spot\nYou just pick a random place on the hill (any point will do).\n\n#### 2. Check the slope\nYou look around and feel which way is downhill. That “feel” is the *gradient* – a number that tells you the steepness and direction of the slope at your current spot.\n\n#### 3. Take a step downhill\nYou step a little bit in the opposite direction of the slope (because you want to go down, not up).  \nThe size of the step is called the **learning rate** – a bit like how fast you walk.\n\n#### 4. Repeat\nNow you’re at a new spot.  \n* Che